# WorldCereal crop type mapping | Guided practical
**ESA Advanced Training Course on Land Remote Sensing (2025)**  
**Learner:** Ibrahim Kabore  
**Environment:** Google Colab, CPU runtime  

## Research question
How accurately can precomputed satellite time-series representations distinguish crop types, and how does performance change when entire reference datasets are withheld?

This independent, annotated practical uses the supplied ESA course data. The archived 2025 notebook URL was unavailable when this practical was prepared; therefore this is an alternative workflow, not a reproduction of its exact code. We will train supervised classifiers on the supplied Presto embeddings and inspect the French inference cube. A full satellite crop map requires that same Presto encoder and compatible preprocessing at inference time; a raw satellite band is not an embedding.

### What you will learn
1. Inspect reference data and satellite extraction tables.
2. Understand Presto embeddings and downstream classification labels.
3. Establish a baseline, then train Random Forest and histogram gradient boosting models.
4. Evaluate grouped generalization and interpret errors.
5. Save reproducible results and inspect the French demonstration satellite cube.

Run one section at a time. Record findings in the reflection prompts; do not interpret expected patterns as measured results.

**References:** [WorldCereal repository and documentation](https://github.com/WorldCereal/worldcereal-classification), [WorldCereal paper](https://doi.org/10.5194/essd-15-5491-2023), [Presto paper](https://arxiv.org/abs/2304.14065).

## 1. Set up the environment
Colab provides a temporary runtime. Google Drive persists our results even when this runtime disconnects. Install only the dependencies that are missing; this practical does not need a GPU because the embeddings have already been computed.

In [ ]:
import importlib.util, subprocess, sys
packages = {'pyarrow':'pyarrow', 'xarray':'xarray', 'netCDF4':'netCDF4', 'sklearn':'scikit-learn', 'rasterio':'rasterio', 'joblib':'joblib', 'seaborn':'seaborn'}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Python:', sys.version.split()[0], '| Dependencies ready')

### 1.1 Import the analysis libraries
Pandas handles reference tables, xarray handles labeled multidimensional satellite arrays, and scikit-learn provides models and validation. Setting a random seed makes model comparisons reproducible.

In [ ]:
from pathlib import Path
import json, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr
import pyarrow.parquet as pq
import sklearn, joblib
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
SEED = 42
sns.set_theme(style='whitegrid')
print('scikit-learn:', sklearn.__version__)

### 1.2 Connect Google Drive
Your four data files stay untouched in `data/`. Copying them to Colab's temporary disk usually makes reading faster; all final artifacts are saved under `outputs/` and `models/` in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ESA_EO_Projects/WorldCereal_Crop_Mapping')
DATA = PROJECT / 'data'
OUTPUT = PROJECT / 'outputs'
MODELS = PROJECT / 'models'
LOCAL = Path('/content/worldcereal_data')
for folder in [OUTPUT, MODELS, LOCAL]:
    folder.mkdir(parents=True, exist_ok=True)
FILES = ['extractions.parquet', 'training_df.parquet', 'processed_training_df.parquet', 'worldcereal-preprocessedinputs-french-demo.nc']
assert all((DATA / f).is_file() for f in FILES), f'Check file names and path: {DATA}'
for f in FILES:
    shutil.copy2(DATA / f, LOCAL / f)
    print(f, f'{(LOCAL / f).stat().st_size/1024**2:.1f} MiB')

## 2. Understand the data lineage
The extraction table contains observations or intermediate satellite data. The training table contains derived features. The processed training table additionally contains `downstream_class`, the label used in this exercise. Inspect the real file schemas before drawing conclusions. Parquet metadata reveals the number of rows without reading every value.

In [ ]:
for name in FILES[:3]:
    pf = pq.ParquetFile(LOCAL / name)
    cols = pf.schema_arrow.names
    print(f'\n{name}: {pf.metadata.num_rows:,} rows; {len(cols)} columns')
    print('First columns:', cols[:12])
    print('Last columns:', cols[-12:])

### 2.1 Load the processed training dataset
Each row is a labeled training sample. The 128 columns beginning with `presto_ft_` are numerical embeddings: compact representations learned by a pretrained model from satellite time series. Their components are not individual spectral bands, and the meaning of one component is generally not directly interpretable.

In [ ]:
df = pd.read_parquet(LOCAL / 'processed_training_df.parquet')
features = sorted([c for c in df.columns if c.startswith('presto_ft_')], key=lambda c: int(c.rsplit('_',1)[-1]))
TARGET = 'downstream_class'
assert len(features) == 128 and TARGET in df.columns
print('Shape:', df.shape, '| Presto features:', len(features))
display(df[[c for c in ['ref_id','sample_id','ewoc_code','year',TARGET] if c in df.columns] + features[:5]].head(10))

### 2.2 Quantify class balance and data quality
Crop classes rarely have equal support. Accuracy alone can be misleading when a dominant class accounts for many samples. We check missing values, repeated sample identifiers, and class frequencies before splitting.

In [ ]:
print('Missing labels:', df[TARGET].isna().sum())
print('Missing feature values:', df[features].isna().sum().sum())
print('Repeated sample IDs:', df['sample_id'].duplicated().sum())
print('Exact duplicate rows:', df.duplicated(subset=['ref_id','sample_id',TARGET]+features).sum())
counts = df[TARGET].value_counts()
display(counts.to_frame('samples'))
ax = counts.sort_values().plot.barh(figsize=(9, max(4, len(counts)*0.35)), color='#267d73')
ax.set(title='Crop class distribution', xlabel='Number of samples', ylabel='Crop type')
plt.tight_layout(); plt.show()

### 2.3 Reference dataset composition
`ref_id` identifies the source reference dataset. Samples from the same source may share collection methods, geography and year. Withholding a whole source is a more demanding evaluation than mixing rows from that source between training and testing. It is not necessarily a spatially independent split: sources can overlap geographically.

In [ ]:
display(df['ref_id'].value_counts().to_frame('samples').head(30))
display(pd.crosstab(df['ref_id'], df[TARGET]).style.background_gradient(cmap='Greens', axis=None))
if 'year' in df.columns:
    display(pd.crosstab(df['year'], df[TARGET]))

**Checkpoint A:** Which crops dominate? How many reference sources exist? Are some crop types present in only one source? Send the class table and reference-source table before continuing if the coverage is very uneven.

## 3. Prepare the supervised learning experiment
Only embedding features enter the model. Geographic/source identifiers, sample IDs, years and the label itself must not be included as predictors. Remove records without labels or source IDs, exact duplicates, and convert infinite feature values to missing values. Imputation will be fitted on training data only.

In [ ]:
work = df.dropna(subset=[TARGET, 'ref_id']).drop_duplicates(subset=['ref_id','sample_id',TARGET]+features).copy()
X = work[features].replace([np.inf, -np.inf], np.nan).astype('float32')
y = work[TARGET].astype(str)
groups = work['ref_id'].astype(str)
labels = sorted(y.unique())
print('Samples:', len(work), '| Classes:', len(labels), '| Reference sources:', groups.nunique())
display(work.groupby(TARGET)['ref_id'].nunique().rename('number_of_sources').sort_values())

### 3.1 Define a held-out reference dataset split
We reserve approximately 25% of reference sources. We try a fixed sequence of candidate splits to obtain coverage of all crop classes in both partitions, using labels only to check coverage (not model performance). If complete coverage is impossible, we explicitly report missing classes rather than silently choosing a random row split. Training-only and test-only classes remain visible in the evaluation.

In [ ]:
assert groups.nunique() >= 2, 'At least two reference sources are required.'
splitter = GroupShuffleSplit(n_splits=100, test_size=0.25, random_state=SEED)
best = None
for tr, te in splitter.split(X, y, groups):
    a, b = set(y.iloc[tr]), set(y.iloc[te])
    score = (len(a & b), -abs(len(te)/len(X)-0.25))
    if best is None or score > best[0]:
        best = (score, tr, te)
    if a == b == set(labels) and abs(len(te)/len(X)-0.25) < 0.1:
        break
_, train_idx, test_idx = best
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train, g_test = groups.iloc[train_idx], groups.iloc[test_idx]
assert not set(g_train) & set(g_test)
print(f'Train: {len(X_train):,} samples, {g_train.nunique()} sources')
print(f'Test:  {len(X_test):,} samples, {g_test.nunique()} sources')
print('Test-only classes:', sorted(set(y_test)-set(y_train)))
print('Train-only classes:', sorted(set(y_train)-set(y_test)))
display(pd.concat([y_train.value_counts().rename('train'), y_test.value_counts().rename('test')], axis=1).fillna(0).astype(int))
print('Held-out sources:', sorted(g_test.unique()))

## 4. Train and compare three classifiers
The majority baseline predicts the most frequent training crop. Random Forest combines multiple decision trees and can capture nonlinear relations among embeddings. Histogram gradient boosting builds trees sequentially to correct earlier errors. We use fixed settings, not test-set hyperparameter optimization. Macro F1 gives each evaluated class equal importance.

In [ ]:
models = {
    'majority_baseline': DummyClassifier(strategy='most_frequent'),
    'random_forest': make_pipeline(SimpleImputer(strategy='median'), RandomForestClassifier(n_estimators=150, min_samples_leaf=2, class_weight='balanced_subsample', n_jobs=2, random_state=SEED)),
    'hist_gradient_boosting': HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=31, learning_rate=0.1, random_state=SEED)
}
predictions, rows = {}, []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    predictions[name] = p
    rows.append({'model':name, 'accuracy':accuracy_score(y_test,p), 'balanced_accuracy':balanced_accuracy_score(y_test,p), 'macro_f1':f1_score(y_test,p,labels=sorted(set(y_test)),average='macro',zero_division=0), 'seconds':time.perf_counter()-start})
results = pd.DataFrame(rows).set_index('model')
display(results.round(4))

### 4.1 Compare results visually
Accuracy measures the overall fraction of correct predictions. Balanced accuracy averages recall across reference-supported classes. Macro F1 averages the class-wise harmonic mean of precision and recall. A class never observed during training cannot be learned by these classifiers; its test results must be interpreted accordingly.

In [ ]:
ax = results[['accuracy','balanced_accuracy','macro_f1']].plot.bar(figsize=(10,5), color=['#386cb0','#7fc97f','#fdc086'])
ax.set(ylim=(0,1), ylabel='Score', title='Held-out reference source evaluation')
plt.xticks(rotation=15, ha='right'); plt.tight_layout()
plt.savefig(OUTPUT/'model_comparison.png', dpi=180, bbox_inches='tight'); plt.show()

## 5. Diagnose errors using the preselected Random Forest
We use Random Forest as our primary model, chosen before examining the test metrics. Gradient boosting is a comparison, not an automatically selected winner. The confusion matrix shows which reference crop types are confused with other crops. Rows correspond to true labels, columns to predicted labels.

In [ ]:
PRIMARY = 'random_forest'
p = predictions[PRIMARY]
report = pd.DataFrame(classification_report(y_test,p,labels=labels,output_dict=True,zero_division=0)).T
display(report.round(3))
fig, ax = plt.subplots(figsize=(max(8,len(labels)*0.65), max(6,len(labels)*0.6)))
ConfusionMatrixDisplay.from_predictions(y_test,p,labels=labels,normalize='true',xticks_rotation=90,values_format='.2f',cmap='Blues',ax=ax)
ax.set_title('Random Forest | Row-normalized confusion matrix')
plt.tight_layout(); plt.savefig(OUTPUT/'confusion_matrix.png',dpi=180,bbox_inches='tight'); plt.show()

### 5.1 Examine errors by origin
A model may perform well on one reference source and poorly on another. Group-level performance is useful for discussing transferability, although very small groups provide unstable estimates.

In [ ]:
errors = work.iloc[test_idx][['ref_id','sample_id',TARGET]].copy()
errors['prediction'] = p
errors['correct'] = errors[TARGET] == errors['prediction']
source_scores = errors.groupby('ref_id').agg(n=('correct','size'),accuracy=('correct','mean')).sort_values('accuracy')
display(source_scores)

**Checkpoint B:** Report the three models' scores, the most confused crop pair, and whether any test-only classes occurred. Do not claim geographical transferability solely from a source-level split.

## 6. Inspect model feature importance
Random Forest impurity importance measures how much each feature contributes to tree splits. This is exploratory: correlated embeddings can distribute importance, and individual embedding dimensions cannot automatically be assigned a biophysical meaning. This chart is not a spectral wavelength importance plot.

In [ ]:
forest = models[PRIMARY].named_steps['randomforestclassifier']
imputer = models[PRIMARY].named_steps['simpleimputer']
used_features = imputer.get_feature_names_out(features)
importance = pd.Series(forest.feature_importances_, index=used_features, name='importance').sort_values(ascending=False)
ax = importance.head(15).sort_values().plot.barh(figsize=(8,5),color='#267d73')
ax.set(title='Top 15 Presto embedding components',xlabel='Mean decrease in impurity')
plt.tight_layout(); plt.savefig(OUTPUT/'embedding_importance.png',dpi=180,bbox_inches='tight'); plt.show()

## 7. Save the experiment
Store the model pipeline (including its imputer), feature ordering, class labels, data split, predictions and evaluation tables. These artifacts are necessary to reproduce and audit your results. Only load joblib models from trusted sources.

In [ ]:
results.to_csv(OUTPUT/'model_comparison.csv')
report.to_csv(OUTPUT/'random_forest_classification_report.csv')
errors.to_csv(OUTPUT/'test_predictions.csv',index=False)
source_scores.to_csv(OUTPUT/'reference_source_scores.csv')
importance.to_csv(OUTPUT/'embedding_importance.csv')
joblib.dump(models[PRIMARY], MODELS/'random_forest_presto.joblib')
manifest = {'seed':SEED,'features':features,'target':TARGET,'classes':labels,'train_sources':sorted(g_train.unique()),'test_sources':sorted(g_test.unique()),'primary_model':PRIMARY,'sklearn_version':sklearn.__version__}
(MODELS/'experiment_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Saved outputs:', sorted(p.name for p in OUTPUT.iterdir()))

## 8. French demonstration area: inspect the satellite data cube
The NetCDF file contains arrays with named dimensions such as time, x and y. We can inspect the coordinates, variable names, resolution and units without loading the entire cube. The file metadata indicates UTM zone 31N (EPSG:32631); verify that in xarray before producing a map. This step is not a crop classification prediction.

In [ ]:
ds = xr.open_dataset(LOCAL / FILES[3])
print(ds)
print('Global attributes:', ds.attrs)
display(pd.DataFrame([{'variable':name,'dimensions':str(v.dims),'shape':str(v.shape),'dtype':str(v.dtype),'units':v.attrs.get('units','not declared')} for name,v in ds.data_vars.items()]))
if 'crs' in ds:
    print(ds['crs'].attrs)

### 8.1 Visualize one spatial variable
Select a variable with both x and y dimensions and take the first index on any remaining dimensions. This is a quick quality-control image, not a true-color composite. Check its name and units before interpreting the colors; the first timestamp may not correspond to a meaningful crop season.

In [ ]:
spatial = [name for name,v in ds.data_vars.items() if {'x','y'}.issubset(v.dims)]
print('Spatial variables:',spatial)
if spatial:
    name = spatial[0]  # change after inspecting the variable table
    v = ds[name]
    image = v.isel({d:0 for d in v.dims if d not in ['x','y']}).transpose('y','x')
    fig,ax = plt.subplots(figsize=(8,6))
    image.plot.imshow(ax=ax,robust=True,cmap='viridis')
    ax.set_title(f'Demo cube: {name} (first slice)')
    plt.tight_layout(); plt.savefig(OUTPUT/'french_demo_first_slice.png',dpi=180,bbox_inches='tight'); plt.show()

### 8.2 Can we apply our classifier directly?
Only if the cube contains the exact same 128 named Presto embedding variables, computed with the same encoder, temporal window and preprocessing. Raw reflectance/time-series values cannot be given to a classifier trained on embeddings. We inspect compatibility; we deliberately do not produce a misleading map from incompatible variables.

In [ ]:
missing_features = sorted(set(features)-set(ds.data_vars))
print('Embedding variables absent:',len(missing_features),'of',len(features))
if missing_features:
    print('The supplied cube needs compatible Presto embedding inference before applying this classifier.')
else:
    print('Names match. Verify encoder, preprocessing, dimensions and season before attempting map inference.')

## 9. Next practical: spatial crop-map generation
The current WorldCereal project provides a classification application for training and applying custom models, including cloud-based processing through openEO. The training head in this notebook is a scikit-learn baseline, not the official WorldCereal custom head format. We must verify the encoder/input schema and the supported head format before building the map-generation notebook.

- [Current official classification application](https://github.com/WorldCereal/worldcereal-classification/blob/main/notebooks/worldcereal_classification_app.ipynb)
- [Official WorldCereal documentation](https://worldcereal.github.io/worldcereal-documentation/)

The next step is to share the printed cube schema and the results from section 4. We can then configure compatible spatial inference and export a georeferenced crop-type map without pretending that the current classifier accepts raw bands.

## 10. Portfolio reflection
Complete these after running the notebook:
- Number of samples and crop classes: ...
- Held-out sources and missing classes: ...
- Random Forest accuracy / balanced accuracy / macro F1: ...
- Largest confusion and plausible explanation: ...
- Limitations (label differences, geographic overlap, class imbalance): ...
- My independent contribution: ...

**Possible CV bullet after verification:** Developed and evaluated crop-type classifiers using 128-dimensional WorldCereal Presto embeddings, reference-source holdout validation, and class-wise performance analysis; documented a reproducible Google Colab workflow. Add measured results only after execution and distinguish this experiment from completed spatial mapping.